# LeetCode 1597 — Build Binary Expression Tree From Infix Expression bằng Python

Link đề: https://leetcode.com/problems/build-binary-expression-tree-from-infix-expression/ · Độ khó: **Medium** (theo danh mục) · Chủ đề: String, Stack, Tree, Binary Tree

> **Ghi chú độ khó:** LeetCode xếp bài này là **Hard**; danh mục của khóa học xếp **Medium** — notebook dùng độ khó của danh mục.
>
> **Ghi chú Premium:** đây là bài Premium. Đề bài được viết lại theo hiểu biết về bài (lấy từ trang doocs: https://leetcode.doocs.org/en/lc/1597/), nên đối chiếu lại nếu bạn có tài khoản Premium.
>
> **Ghi chú chữ ký:** trang doocs không có code Python cho bài này. Tên method `expTree` lấy từ lời giải C++ công khai (walkccc/LeetCode, `Node* expTree(string s)`); lớp `Node` dưới đây viết theo hiểu biết — cần đối chiếu nếu có tài khoản Premium.

Skeleton (dự kiến):

```python
# Definition for a binary tree node.
# class Node(object):
#     def __init__(self, val=" ", left=None, right=None):
#         self.val = val
#         self.left = left
#         self.right = right
class Solution:
    def expTree(self, s: str) -> 'Node':
        ...
```

## 1. Mục tiêu học

1. Hiểu **cây biểu thức nhị phân**: lá là toán hạng, nút trong là toán tử có đúng 2 con; duyệt **trung thứ tự (inorder)** cho lại biểu thức không ngoặc.
2. Nhận ra rằng "inorder đúng" là **chưa đủ**: cây còn phải tôn trọng **thứ tự ưu tiên** (`*`, `/` trước `+`, `-`), **ngoặc**, và **kết hợp trái** (`1-2-3 = (1-2)-3`).
3. Viết 3 lời giải chính: **chia để trị theo toán tử ưu tiên thấp nhất bên phải nhất**, **phân tích đệ quy (recursive descent) expr/term/factor**, và **hai ngăn xếp (shunting-yard)**.
4. Viết **hàm kiểm tra hợp lệ** (vì bài có nhiều đáp án): inorder bỏ ngoặc, mọi nút trong có 2 con, giá trị bằng `Fraction`, toán hạng giữ thứ tự — và hiểu giới hạn của kiểm tra theo giá trị.
5. Dựng được bộ test ngẫu nhiên sinh biểu thức hợp lệ và đối chiếu các cách với nhau.

## 2. Đề bài & ví dụ

**Cây biểu thức nhị phân** là cây nhị phân biểu diễn biểu thức số học: mỗi nút có **0 hoặc 2 con**. **Lá** là toán hạng (số), **nút trong** là một trong các toán tử `'+'`, `'-'`, `'*'`, `'/'`. Nút trong với toán tử `o` biểu diễn biểu thức `(A o B)`, với `A` là biểu thức của cây con trái, `B` của cây con phải.

Cho chuỗi `s` là **biểu thức trung tố** gồm toán hạng, các toán tử trên và ngoặc `'('`, `')'`. Trả về **một cây biểu thức hợp lệ bất kỳ** sao cho **duyệt inorder** của cây cho lại đúng `s` **sau khi bỏ các dấu ngoặc**.

Lưu ý: **thứ tự phép tính áp dụng** cho `s` — trong ngoặc tính trước, nhân/chia trước cộng/trừ. Toán hạng phải xuất hiện **cùng thứ tự** trong `s` và trong inorder của cây.

**Ví dụ 1:** `s = "3*4-2*5"` → `[-,*,*,3,4,2,5]` (cây duy nhất hợp lệ).

**Ví dụ 2:** `s = "2-3/(5*2)+1"` → `[+,-,1,2,/,null,null,null,null,3,*,null,null,5,2]`. Inorder của cây là `2-3/5*2+1` = `s` bỏ ngoặc. Có những cây khác cùng inorder nhưng **tính ra giá trị khác** ⇒ không hợp lệ; cũng có cây cùng giá trị nhưng inorder khác / thứ tự toán hạng khác ⇒ không hợp lệ.

**Ví dụ 3:** `s = "1+2+3+4+5"` → `[+,+,5,+,4,null,null,+,3,null,null,1,2]`. Cây `[+,+,5,+,+,null,null,1,2,3,4]` cũng là một trong nhiều đáp án hợp lệ.

**Ràng buộc:**
- `1 ≤ s.length ≤ 100`
- `s` chỉ gồm chữ số và các ký tự `'('`, `')'`, `'+'`, `'-'`, `'*'`, `'/'`.
- Mỗi toán hạng có **đúng 1 chữ số**.
- `s` được đảm bảo là biểu thức hợp lệ.

In [1]:
import random
import time
import re
from fractions import Fraction
from collections import deque

class Node(object):
    """Lớp nút theo hiểu biết về đề (cần đối chiếu Premium)."""
    def __init__(self, val=" ", left=None, right=None):
        self.val = val
        self.left = left
        self.right = right

examples = [
    ("3*4-2*5", "[-,*,*,3,4,2,5]"),
    ("2-3/(5*2)+1", "[+,-,1,2,/,null,null,null,null,3,*,null,null,5,2]"),
    ("1+2+3+4+5", "[+,+,5,+,4,null,null,+,3,null,null,1,2]"),
]
for s, out in examples:
    print(f"s={s!r:16} bỏ ngoặc: {s.replace('(', '').replace(')', '')!r:14} một đáp án mẫu: {out}")

s='3*4-2*5'        bỏ ngoặc: '3*4-2*5'      một đáp án mẫu: [-,*,*,3,4,2,5]
s='2-3/(5*2)+1'    bỏ ngoặc: '2-3/5*2+1'    một đáp án mẫu: [+,-,1,2,/,null,null,null,null,3,*,null,null,5,2]
s='1+2+3+4+5'      bỏ ngoặc: '1+2+3+4+5'    một đáp án mẫu: [+,+,5,+,4,null,null,+,3,null,null,1,2]


## 3. Kiến thức nền

### 3.1 Cây nhị phân, inorder và in theo tầng (level-order)

- **Inorder** (trái – gốc – phải): với cây biểu thức cho ra biểu thức trung tố **không ngoặc**.
- **Level-order** (BFS theo tầng) với `null` cho chỗ trống là cách LeetCode in cây. Ta viết `tree_to_list` để so với đầu ra mẫu (bỏ các `null` thừa ở cuối).

In [2]:
def inorder(root):
    if root is None:
        return ""
    return inorder(root.left) + root.val + inorder(root.right)

def tree_to_list(root):
    out, q = [], deque([root])
    while q:
        node = q.popleft()
        if node is None:
            out.append("null"); continue
        out.append(node.val)
        q.append(node.left); q.append(node.right)
    while out and out[-1] == "null":
        out.pop()
    return "[" + ",".join(out) + "]"

# Dựng tay cây ví dụ 1:   -
#                       *   *
#                      3 4 2 5
t = Node("-", Node("*", Node("3"), Node("4")), Node("*", Node("2"), Node("5")))
print("inorder:", inorder(t))
print("level-order:", tree_to_list(t))

inorder: 3*4-2*5
level-order: [-,*,*,3,4,2,5]


### 3.2 Tính giá trị cây bằng `Fraction`

Phép chia `/` trong đề là chia **thực** (không phải chia nguyên). Dùng `fractions.Fraction` để tính **chính xác** (không sai số dấu phẩy động). Chia cho 0 thì ta trả `None` để so sánh vẫn được.

In [3]:
def eval_tree(root):
    if root.left is None and root.right is None:
        return Fraction(int(root.val))
    a, b = eval_tree(root.left), eval_tree(root.right)
    if a is None or b is None:
        return None
    if root.val == "+": return a + b
    if root.val == "-": return a - b
    if root.val == "*": return a * b
    if b == 0: return None
    return a / b

print(eval_tree(t), Fraction(1, 3) + Fraction(1, 6))

2 1/2


### 3.3 Giá trị "chuẩn" của biểu thức (oracle độc lập)

Để kiểm tra mà **không phụ thuộc** vào lời giải của ta, dùng chính trình thông dịch Python: thay mỗi chữ số `d` bằng `F(d)` (Fraction) rồi `eval`. Chuỗi đầu vào chỉ gồm chữ số, toán tử và ngoặc (ta tự sinh), nên dùng `eval` ở đây là an toàn và Python đã cài đúng thứ tự ưu tiên + kết hợp trái.

In [4]:
def ref_value(s, values=None):
    """Giá trị chuẩn của s. values: list Fraction thay cho từng toán hạng theo thứ tự (để kiểm tra mạnh)."""
    it = iter(values) if values is not None else None
    expr = re.sub(r"\d", lambda m: f"F({m.group()})" if it is None else (lambda v: f"F({v.numerator},{v.denominator})")(next(it)), s)
    try:
        return eval(expr, {"F": Fraction, "__builtins__": {}})
    except ZeroDivisionError:
        return None

print(ref_value("3*4-2*5"), ref_value("2-3/(5*2)+1"), ref_value("1-2-3"), ref_value("8/2/2"), ref_value("1/(2-2)"))

2 27/10 -4 2 None


### 3.4 Ngăn xếp (`list` làm stack) và độ ưu tiên toán tử

`append` / `pop` ở cuối `list` là O(1). Bảng ưu tiên: `+`, `-` mức 1; `*`, `/` mức 2. Tất cả đều **kết hợp trái**: khi gặp toán tử cùng mức, toán tử đến trước được "chốt" trước.

In [5]:
PREC = {"+": 1, "-": 1, "*": 2, "/": 2}
st = []
for c in "abc":
    st.append(c)
print(st, "-> pop:", st.pop(), st)
print("PREC['*'] > PREC['-'] ?", PREC["*"] > PREC["-"])

['a', 'b'] -> pop: c ['a', 'b']
PREC['*'] > PREC['-'] ? True


### 3.5 Ghép cặp ngoặc bằng stack

`match[i]` = chỉ số ngoặc đóng ứng với ngoặc mở tại `i`. Dùng cho cách chia để trị.

In [6]:
def match_parens(s):
    match, st = {}, []
    for i, c in enumerate(s):
        if c == "(":
            st.append(i)
        elif c == ")":
            j = st.pop(); match[j] = i; match[i] = j
    return match

print(match_parens("2-3/(5*2)+1"), match_parens("((1+2))*3"))

{4: 8, 8: 4} {1: 5, 5: 1, 0: 6, 6: 0}


## 4. Ý tưởng nền tảng / trực giác

Xét `s = "3*4-2*5"`. Biểu thức bằng `(3*4) - (2*5)`: phép tính **thực hiện sau cùng** là `-` ⇒ nó phải là **gốc**. Hai bên là `3*4` và `2*5` ⇒ hai cây con.

**Quy luật:** gốc của (một đoạn) biểu thức là toán tử **được tính cuối cùng**, tức là:
1. nằm **ngoài mọi ngoặc** của đoạn đó,
2. có **độ ưu tiên thấp nhất** (`+`/`-` thấp hơn `*`/`/`),
3. nếu nhiều toán tử cùng mức thấp nhất: chọn cái **bên phải nhất** (vì kết hợp trái: `1-2-3 = (1-2)-3`, phép trừ thứ hai tính sau cùng).

Nếu cả đoạn được bọc trong một cặp ngoặc thì bỏ ngoặc ngoài rồi làm lại; nếu đoạn chỉ là một chữ số thì đó là lá.

**Tại sao "inorder đúng" chưa đủ?** Với cùng chuỗi `3*4-2*5`, cây `((3*4)-2)*5` (gốc `*` cuối) cũng có inorder `3*4-2*5` nhưng giá trị là `50`, không phải `2`. Đề yêu cầu cây phải **tính đúng**.

In [7]:
good = Node("-", Node("*", Node("3"), Node("4")), Node("*", Node("2"), Node("5")))
bad = Node("*", Node("-", Node("*", Node("3"), Node("4")), Node("2")), Node("5"))
for name, tr in [("đúng", good), ("sai ưu tiên", bad)]:
    print(f"{name:12s} inorder={inorder(tr)}  giá trị={eval_tree(tr)}  level={tree_to_list(tr)}")
print("giá trị chuẩn của s:", ref_value("3*4-2*5"))

đúng         inorder=3*4-2*5  giá trị=2  level=[-,*,*,3,4,2,5]
sai ưu tiên  inorder=3*4-2*5  giá trị=50  level=[*,-,5,*,2,null,null,3,4]
giá trị chuẩn của s: 2


### 4.1 Hàm kiểm tra hợp lệ (dùng cho mọi test)

Vì có nhiều đáp án, ta **không so sánh với một cây cố định** mà kiểm tra:
1. Mọi nút trong có **đúng 2 con** và là toán tử; mọi lá là **chữ số**.
2. `inorder(cây) == s bỏ ngoặc` (kéo theo toán hạng **giữ thứ tự**, ta vẫn kiểm riêng cho rõ).
3. Giá trị cây (`Fraction`) == giá trị chuẩn của `s`.

**Giới hạn:** kiểm theo giá trị có thể "nhầm may mắn": với `s = "1*1+1"`, cây `1*(1+1)` cũng cho 2. Vì vậy ta thêm **kiểm tra mạnh**: thay các toán hạng bằng các phân số ngẫu nhiên (cùng thay ở cả cây lẫn biểu thức) rồi so giá trị nhiều lần. Cây đúng cấu trúc thì luôn khớp; cây sai cấu trúc gần như chắc chắn lệch. (Các cây khác nhau chỉ do tính kết hợp của `+`/`*`, như ví dụ 3, vẫn được chấp nhận — đúng như đề.)

In [8]:
OPS = set("+-*/")

def leaves(root):
    if root.left is None and root.right is None:
        return [root]
    return leaves(root.left) + leaves(root.right)

def is_full(root):
    if root is None:
        return False
    if root.left is None and root.right is None:
        return root.val.isdigit() and len(root.val) == 1
    return root.val in OPS and is_full(root.left) and is_full(root.right)

def eval_tree_with(root, mapping):
    """Tính cây, thay lá bằng mapping[id(lá)]."""
    if root.left is None and root.right is None:
        return mapping[id(root)]
    a, b = eval_tree_with(root.left, mapping), eval_tree_with(root.right, mapping)
    if a is None or b is None: return None
    if root.val == "+": return a + b
    if root.val == "-": return a - b
    if root.val == "*": return a * b
    return None if b == 0 else a / b

def valid_basic(s, root):
    plain = s.replace("(", "").replace(")", "")
    if not is_full(root) or inorder(root) != plain:
        return False
    if [lf.val for lf in leaves(root)] != [c for c in s if c.isdigit()]:
        return False
    return eval_tree(root) == ref_value(s)

def valid_strong(s, root, trials=6, seed=12345):
    if not valid_basic(s, root):
        return False
    rnd = random.Random(seed)
    lv = leaves(root)
    for _ in range(trials):
        vals = [Fraction(rnd.randint(1, 97), rnd.randint(1, 13)) for _ in lv]
        mapping = {id(lf): v for lf, v in zip(lv, vals)}
        if eval_tree_with(root, mapping) != ref_value(s, vals):
            return False
    return True

lucky = Node("*", Node("1"), Node("+", Node("1"), Node("1")))      # 1*(1+1) cho s = "1*1+1"
print("cây 1*(1+1) với s='1*1+1': basic =", valid_basic("1*1+1", lucky), "| strong =", valid_strong("1*1+1", lucky))
print("cây đúng ví dụ 1: basic =", valid_basic("3*4-2*5", good), "| strong =", valid_strong("3*4-2*5", good))
print("cây sai ưu tiên : basic =", valid_basic("3*4-2*5", bad))

cây 1*(1+1) với s='1*1+1': basic = True | strong = False
cây đúng ví dụ 1: basic = True | strong = True
cây sai ưu tiên : basic = False


## 5. Cách 1 — Brute Force / baseline: thử mọi cây trên dãy token

**Ý tưởng:** bỏ ngoặc khỏi `s` được dãy `d₀ o₁ d₁ o₂ … o_k d_k`. Mọi cây có inorder đúng bằng dãy này tương ứng với một cách "đặt ngoặc đầy đủ" — sinh tất cả (số lượng là số Catalan `C_k`) và chọn cây đầu tiên **qua được kiểm tra mạnh**.

Đây là baseline hiểu-đề: nó dùng định nghĩa "hợp lệ" trực tiếp, không cần hiểu thứ tự ưu tiên. Nhưng số cây tăng theo hàm mũ.

In [9]:
def all_trees(tokens, i, j):
    """Mọi cây cho các toán hạng tokens[i], tokens[i+2], ..., tokens[j] (i, j chẵn)."""
    if i == j:
        return [Node(tokens[i])]
    res = []
    for k in range(i + 1, j, 2):                  # k là vị trí toán tử làm gốc
        for L in all_trees(tokens, i, k - 1):
            for R in all_trees(tokens, k + 1, j):
                res.append(Node(tokens[k], L, R))
    return res

def exp_tree_brute(s):
    tokens = [c for c in s if c not in "()"]
    for tr in all_trees(tokens, 0, len(tokens) - 1):
        if valid_strong(s, tr):
            return tr
    return None

for s, _ in examples:
    tr = exp_tree_brute(s)
    print(f"{s:14s} -> {tree_to_list(tr)}  hợp lệ: {valid_strong(s, tr)}")

3*4-2*5        -> [-,*,*,3,4,2,5]  hợp lệ: True
2-3/(5*2)+1    -> [+,-,1,2,/,null,null,null,null,3,*,null,null,5,2]  hợp lệ: True
1+2+3+4+5      -> [+,1,+,null,null,2,+,null,null,3,+,null,null,4,5]  hợp lệ: True


**Độ phức tạp:** với `k` toán tử, số cây là `C_k = (2k)! / ((k+1)! k!)` ≈ `4^k / k^1.5`; mỗi cây kiểm tra O(n). Với `s` dài 100 (≈ 50 toán tử) là bất khả thi.

**Nút thắt:** ta sinh rất nhiều cây sai thứ tự ưu tiên rồi mới loại. Cần **dựng thẳng** cây đúng.

In [10]:
from math import comb
for k in [3, 6, 9, 12, 49]:
    print(f"{k:2d} toán tử -> {comb(2 * k, k) // (k + 1):,} cây")
t0 = time.perf_counter(); exp_tree_brute("1-2*3+4/5-6*7"); print(f"brute 6 toán tử: {time.perf_counter() - t0:.3f}s")

 3 toán tử -> 5 cây
 6 toán tử -> 132 cây
 9 toán tử -> 4,862 cây
12 toán tử -> 208,012 cây
49 toán tử -> 509,552,245,179,617,138,054,608,572 cây
brute 6 toán tử: 0.004s


## 6. Ý tưởng thuật toán / bước đột phá

Có hai cách nhìn đều dẫn tới thuật toán tuyến tính hoặc gần tuyến tính:

**(a) Từ trên xuống — tìm gốc:** gốc là toán tử **tính cuối cùng** = ngoài ngoặc, ưu tiên thấp nhất, bên phải nhất. Đệ quy cho hai nửa.

**(b) Văn phạm (grammar):**
```
expr   := term   (('+' | '-') term)*      # cộng/trừ, kết hợp trái
term   := factor (('*' | '/') factor)*    # nhân/chia, kết hợp trái
factor := digit | '(' expr ')'
```
Mỗi quy tắc thành một hàm; vòng `while` trong `expr`/`term` dựng cây **lệch trái** (`node = Node(op, node, next)`) ⇒ tự động kết hợp trái.

**(c) Hai ngăn xếp:** đọc từ trái sang phải, giữ stack **nút** và stack **toán tử**. Quy tắc "nếu… thì…":
- Nếu gặp chữ số thì đẩy một lá vào stack nút.
- Nếu gặp `(` thì đẩy vào stack toán tử.
- Nếu gặp `)` thì "chốt" (pop toán tử, ghép 2 nút) cho đến khi gặp `(`, rồi bỏ `(`.
- Nếu gặp toán tử `o` thì: **chừng nào** đỉnh stack là toán tử có ưu tiên **≥** `o` (dấu `≥` tạo kết hợp trái), chốt nó; rồi đẩy `o`.
- Hết chuỗi thì chốt toàn bộ. Nút còn lại là gốc.

**Invariant (c):** stack toán tử (giữa hai dấu `(`) luôn có ưu tiên **tăng nghiêm ngặt** từ đáy lên đỉnh; mỗi lần chốt tạo đúng một nút trong có 2 con.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Chia để trị: toán tử ưu tiên thấp nhất, bên phải nhất, ngoài ngoặc

Từng đoạn:
1. Tiền xử lý `match` cặp ngoặc.
2. `build(l, r)` dựng cây cho `s[l:r]`: nếu `s[l]` là `(` và ngoặc đóng tương ứng ở `r−1` ⇒ bỏ cặp ngoặc bọc ngoài (lặp lại).
3. Nếu còn 1 ký tự ⇒ lá.
4. Quét đoạn, theo dõi `depth`; ở `depth == 0` ghi lại vị trí `+`/`-` **cuối cùng** và `*`/`/` **cuối cùng**.
5. Gốc là `+`/`-` cuối nếu có, ngược lại `*`/`/` cuối. Đệ quy hai bên.

In [11]:
def exp_tree_split(s):
    match = match_parens(s)

    def build(l, r):
        while s[l] == "(" and match[l] == r - 1:      # bỏ ngoặc bọc ngoài
            l += 1; r -= 1
        if r - l == 1:
            return Node(s[l])
        depth, pos_add, pos_mul = 0, -1, -1
        for k in range(l, r):
            c = s[k]
            if c == "(":
                depth += 1
            elif c == ")":
                depth -= 1
            elif depth == 0:
                if c in "+-":
                    pos_add = k                      # giữ cái bên phải nhất
                elif c in "*/":
                    pos_mul = k
        k = pos_add if pos_add != -1 else pos_mul
        return Node(s[k], build(l, k), build(k + 1, r))

    return build(0, len(s))

for s, _ in examples + [("((1+2))*3", ""), ("1-2-3", ""), ("8/2/2", "")]:
    tr = exp_tree_split(s)
    print(f"{s:14s} -> {tree_to_list(tr):52s} hợp lệ: {valid_strong(s, tr)}")

3*4-2*5        -> [-,*,*,3,4,2,5]                                      hợp lệ: True
2-3/(5*2)+1    -> [+,-,1,2,/,null,null,null,null,3,*,null,null,5,2]    hợp lệ: True
1+2+3+4+5      -> [+,+,5,+,4,null,null,+,3,null,null,1,2]              hợp lệ: True
((1+2))*3      -> [*,+,3,1,2]                                          hợp lệ: True
1-2-3          -> [-,-,3,1,2]                                          hợp lệ: True
8/2/2          -> [/,/,2,8,2]                                          hợp lệ: True


Code nộp LeetCode của Cách 2:

```python
class Solution:
    def expTree(self, s: str) -> 'Node':
        match, st = {}, []
        for i, c in enumerate(s):
            if c == "(":
                st.append(i)
            elif c == ")":
                match[st.pop()] = i
        def build(l, r):
            while s[l] == "(" and match[l] == r - 1:
                l += 1; r -= 1
            if r - l == 1:
                return Node(s[l])
            depth, pos_add, pos_mul = 0, -1, -1
            for k in range(l, r):
                c = s[k]
                if c == "(": depth += 1
                elif c == ")": depth -= 1
                elif depth == 0:
                    if c in "+-": pos_add = k
                    elif c in "*/": pos_mul = k
            k = pos_add if pos_add != -1 else pos_mul
            return Node(s[k], build(l, k), build(k + 1, r))
        return build(0, len(s))
```

**Độ phức tạp:** mỗi tầng đệ quy quét đoạn của nó ⇒ worst case O(n²) (ví dụ `1+2+3+…` tạo cây lệch, mỗi lần chỉ bớt 2 ký tự); với `n ≤ 100` hoàn toàn ổn. Space O(n) (đệ quy + `match`).

### 7.2 Cách 3 — Phân tích đệ quy (recursive descent)

Một con trỏ `i` chạy dọc chuỗi, ba hàm lồng nhau:
- `factor()`: nếu `s[i]` là `(` ⇒ bỏ qua `(`, gọi `expr()`, bỏ qua `)`; ngược lại tạo lá từ chữ số.
- `term()`: lấy một `factor`, rồi **chừng nào** gặp `*` hoặc `/` thì ghép `Node(op, node, factor())`.
- `expr()`: lấy một `term`, rồi chừng nào gặp `+`/`-` thì ghép `Node(op, node, term())`.

Mỗi ký tự được đọc đúng một lần ⇒ O(n).

In [12]:
def exp_tree_rd(s):
    n, i = len(s), 0

    def expr():
        nonlocal i
        node = term()
        while i < n and s[i] in "+-":
            op = s[i]; i += 1
            node = Node(op, node, term())        # lệch trái = kết hợp trái
        return node

    def term():
        nonlocal i
        node = factor()
        while i < n and s[i] in "*/":
            op = s[i]; i += 1
            node = Node(op, node, factor())
        return node

    def factor():
        nonlocal i
        if s[i] == "(":
            i += 1                               # bỏ '('
            node = expr()
            i += 1                               # bỏ ')'
            return node
        node = Node(s[i]); i += 1
        return node

    return expr()

for s, _ in examples:
    tr = exp_tree_rd(s)
    print(f"{s:14s} -> {tree_to_list(tr):52s} hợp lệ: {valid_strong(s, tr)}")

3*4-2*5        -> [-,*,*,3,4,2,5]                                      hợp lệ: True
2-3/(5*2)+1    -> [+,-,1,2,/,null,null,null,null,3,*,null,null,5,2]    hợp lệ: True
1+2+3+4+5      -> [+,+,5,+,4,null,null,+,3,null,null,1,2]              hợp lệ: True


Với ví dụ 1 và 2, Cách 3 cho **đúng** cây mẫu của đề. Với ví dụ 3, nó cho cây lệch trái `[+,+,5,+,4,null,null,+,3,null,null,1,2]` — cũng chính là đầu ra mẫu.

Code nộp LeetCode của Cách 3:

```python
class Solution:
    def expTree(self, s: str) -> 'Node':
        n, i = len(s), 0
        def expr():
            nonlocal i
            node = term()
            while i < n and s[i] in "+-":
                op = s[i]; i += 1
                node = Node(op, node, term())
            return node
        def term():
            nonlocal i
            node = factor()
            while i < n and s[i] in "*/":
                op = s[i]; i += 1
                node = Node(op, node, factor())
            return node
        def factor():
            nonlocal i
            if s[i] == "(":
                i += 1
                node = expr()
                i += 1
                return node
            node = Node(s[i]); i += 1
            return node
        return expr()
```

**Độ phức tạp:** Time O(n), Space O(n) (độ sâu đệ quy tỉ lệ với độ lồng ngoặc).

### 7.3 Cách 4 — Hai ngăn xếp (shunting-yard dựng nút)

Đây là thuật toán "sân ga" (shunting-yard) của Dijkstra, nhưng thay vì xuất ra hậu tố ta **dựng nút** ngay khi chốt một toán tử. Không đệ quy ⇒ không lo giới hạn độ sâu.

In [13]:
def exp_tree_stacks(s):
    prec = {"+": 1, "-": 1, "*": 2, "/": 2}
    nodes, ops = [], []

    def reduce_once():
        op = ops.pop()
        right = nodes.pop(); left = nodes.pop()   # chú ý thứ tự: phải pop trước
        nodes.append(Node(op, left, right))

    for c in s:
        if c.isdigit():
            nodes.append(Node(c))
        elif c == "(":
            ops.append(c)
        elif c == ")":
            while ops[-1] != "(":
                reduce_once()
            ops.pop()                             # bỏ '('
        else:
            while ops and ops[-1] != "(" and prec[ops[-1]] >= prec[c]:
                reduce_once()
            ops.append(c)
    while ops:
        reduce_once()
    return nodes[-1]

for s, _ in examples:
    tr = exp_tree_stacks(s)
    print(f"{s:14s} -> {tree_to_list(tr):52s} hợp lệ: {valid_strong(s, tr)}")

3*4-2*5        -> [-,*,*,3,4,2,5]                                      hợp lệ: True
2-3/(5*2)+1    -> [+,-,1,2,/,null,null,null,null,3,*,null,null,5,2]    hợp lệ: True
1+2+3+4+5      -> [+,+,5,+,4,null,null,+,3,null,null,1,2]              hợp lệ: True


Code nộp LeetCode của Cách 4: xem mục 9 (đây là cách được chọn để nộp).

**Độ phức tạp:** mỗi ký tự được đẩy và lấy ra tối đa một lần ⇒ Time O(n), Space O(n).

## 8. TRACE từng bước

Trace Cách 4 trên `s = "2-3/(5*2)+1"` (biểu diễn nút bằng biểu thức có ngoặc đầy đủ cho dễ đọc):

| ký tự | hành động | stack nút | stack toán tử |
|---|---|---|---|
| 2 | lá | 2 | |
| - | đẩy | 2 | - |
| 3 | lá | 2, 3 | - |
| / | `-` có ưu tiên thấp hơn ⇒ không chốt; đẩy | 2, 3 | - / |
| ( | đẩy | 2, 3 | - / ( |
| 5 | lá | 2, 3, 5 | - / ( |
| * | đỉnh là `(` ⇒ đẩy | 2, 3, 5 | - / ( * |
| 2 | lá | 2, 3, 5, 2 | - / ( * |
| ) | chốt `*` ⇒ (5*2); bỏ `(` | 2, 3, (5*2) | - / |
| + | chốt `/` ⇒ (3/(5*2)); chốt `-` ⇒ (2-(3/(5*2))); đẩy | (2-…) | + |
| 1 | lá | (2-…), 1 | + |
| hết | chốt `+` | ((2-(3/(5*2)))+1) | |

In [14]:
def to_paren(node):
    if node.left is None:
        return node.val
    return "(" + to_paren(node.left) + node.val + to_paren(node.right) + ")"

def trace_stacks(s):
    prec = {"+": 1, "-": 1, "*": 2, "/": 2}
    nodes, ops = [], []
    def reduce_once(log):
        op = ops.pop(); right = nodes.pop(); left = nodes.pop()
        nodes.append(Node(op, left, right)); log.append(f"chốt {op}")
    for c in s:
        log = []
        if c.isdigit():
            nodes.append(Node(c)); log.append("lá")
        elif c == "(":
            ops.append(c); log.append("đẩy (")
        elif c == ")":
            while ops[-1] != "(":
                reduce_once(log)
            ops.pop(); log.append("bỏ (")
        else:
            while ops and ops[-1] != "(" and prec[ops[-1]] >= prec[c]:
                reduce_once(log)
            ops.append(c); log.append(f"đẩy {c}")
        print(f"{c!r:4} {', '.join(log):28s} nút={[to_paren(x) for x in nodes]}  ops={ops}")
    log = []
    while ops:
        reduce_once(log)
    print(f"hết  {', '.join(log):28s} nút={[to_paren(x) for x in nodes]}")
    return nodes[-1]

root = trace_stacks("2-3/(5*2)+1")
print("level-order:", tree_to_list(root))

'2'  lá                           nút=['2']  ops=[]
'-'  đẩy -                        nút=['2']  ops=['-']
'3'  lá                           nút=['2', '3']  ops=['-']
'/'  đẩy /                        nút=['2', '3']  ops=['-', '/']
'('  đẩy (                        nút=['2', '3']  ops=['-', '/', '(']
'5'  lá                           nút=['2', '3', '5']  ops=['-', '/', '(']
'*'  đẩy *                        nút=['2', '3', '5']  ops=['-', '/', '(', '*']
'2'  lá                           nút=['2', '3', '5', '2']  ops=['-', '/', '(', '*']
')'  chốt *, bỏ (                 nút=['2', '3', '(5*2)']  ops=['-', '/']
'+'  chốt /, chốt -, đẩy +        nút=['(2-(3/(5*2)))']  ops=['+']
'1'  lá                           nút=['(2-(3/(5*2)))', '1']  ops=['+']
hết  chốt +                       nút=['((2-(3/(5*2)))+1)']
level-order: [+,-,1,2,/,null,null,null,null,3,*,null,null,5,2]


In [15]:
# Trace Cách 3 (recursive descent): in ra mỗi lần vào hàm và mỗi lần ghép nút
def trace_rd(s):
    n, i = len(s), 0
    def expr(d):
        nonlocal i
        print("  " * d + f"expr  tại i={i}")
        node = term(d + 1)
        while i < n and s[i] in "+-":
            op = s[i]; i += 1
            r = term(d + 1); node = Node(op, node, r)
            print("  " * d + f"expr  ghép -> {to_paren(node)}")
        return node
    def term(d):
        nonlocal i
        node = factor(d + 1)
        while i < n and s[i] in "*/":
            op = s[i]; i += 1
            r = factor(d + 1); node = Node(op, node, r)
            print("  " * d + f"term  ghép -> {to_paren(node)}")
        return node
    def factor(d):
        nonlocal i
        if s[i] == "(":
            print("  " * d + f"factor '(' tại i={i}")
            i += 1; node = expr(d + 1); i += 1
            return node
        node = Node(s[i]); i += 1
        return node
    return expr(0)

print(tree_to_list(trace_rd("3*4-2*5")))

expr  tại i=0
  term  ghép -> (3*4)
  term  ghép -> (2*5)
expr  ghép -> ((3*4)-(2*5))
[-,*,*,3,4,2,5]


## 9. Code nộp LeetCode

Chọn Cách 4 (hai ngăn xếp): O(n), không đệ quy. Lớp `Node` được LeetCode cung cấp sẵn (ở đây ta đã tự định nghĩa để chạy thử — **cần đối chiếu Premium**).

```python
# class Node(object):
#     def __init__(self, val=" ", left=None, right=None):
#         self.val = val
#         self.left = left
#         self.right = right
class Solution:
    def expTree(self, s: str) -> 'Node':
        prec = {'+': 1, '-': 1, '*': 2, '/': 2}
        nodes, ops = [], []

        def reduce_once():
            op = ops.pop()
            right = nodes.pop()
            left = nodes.pop()
            nodes.append(Node(op, left, right))

        for c in s:
            if c.isdigit():
                nodes.append(Node(c))
            elif c == '(':
                ops.append(c)
            elif c == ')':
                while ops[-1] != '(':
                    reduce_once()
                ops.pop()
            else:
                while ops and ops[-1] != '(' and prec[ops[-1]] >= prec[c]:
                    reduce_once()
                ops.append(c)
        while ops:
            reduce_once()
        return nodes[-1]
```

In [16]:
class Solution:
    def expTree(self, s: str) -> 'Node':
        prec = {'+': 1, '-': 1, '*': 2, '/': 2}
        nodes, ops = [], []

        def reduce_once():
            op = ops.pop()
            right = nodes.pop()
            left = nodes.pop()
            nodes.append(Node(op, left, right))

        for c in s:
            if c.isdigit():
                nodes.append(Node(c))
            elif c == '(':
                ops.append(c)
            elif c == ')':
                while ops[-1] != '(':
                    reduce_once()
                ops.pop()
            else:
                while ops and ops[-1] != '(' and prec[ops[-1]] >= prec[c]:
                    reduce_once()
                ops.append(c)
        while ops:
            reduce_once()
        return nodes[-1]

for s, sample in examples:
    tr = Solution().expTree(s)
    print(f"{s:14s} -> {tree_to_list(tr)}  hợp lệ: {valid_strong(s, tr)}  trùng mẫu: {tree_to_list(tr) == sample}")

3*4-2*5        -> [-,*,*,3,4,2,5]  hợp lệ: True  trùng mẫu: True
2-3/(5*2)+1    -> [+,-,1,2,/,null,null,null,null,3,*,null,null,5,2]  hợp lệ: True  trùng mẫu: True
1+2+3+4+5      -> [+,+,5,+,4,null,null,+,3,null,null,1,2]  hợp lệ: True  trùng mẫu: True


## 10. Vì sao thuật toán đúng

**Cách 2 (chia để trị):** trong một đoạn biểu thức, toán tử được tính **cuối cùng** là toán tử ngoài ngoặc, ưu tiên thấp nhất, và (do kết hợp trái) bên phải nhất trong số đó. Mọi toán tử khác thuộc hẳn về nửa trái hoặc nửa phải của nó, và hai nửa đó cũng là biểu thức hợp lệ được tính **độc lập** trước. Vì vậy `Node(op, cây(trái), cây(phải))` biểu diễn đúng biểu thức (quy nạp theo độ dài). Inorder giữ nguyên thứ tự ký tự (bỏ ngoặc) vì ta chỉ chia đoạn tại vị trí `k`, không đảo gì.

**Cách 3 (văn phạm):** văn phạm `expr/term/factor` là văn phạm chuẩn mô tả đúng ưu tiên (nhân/chia "dính chặt" hơn vì nằm sâu hơn) và kết hợp trái (vòng lặp gộp dần sang trái). Mỗi hàm tiêu thụ đúng phần chuỗi của quy tắc nó biểu diễn.

**Cách 4 (hai stack):** bất biến: các toán tử trong stack (giữa hai `(`) có ưu tiên tăng nghiêm ngặt. Khi gặp `o`, mọi toán tử trên đỉnh có ưu tiên `≥ o` **phải** được tính trước `o` (ưu tiên cao hơn, hoặc cùng mức mà đến trước — kết hợp trái), nên chốt chúng là đúng. Ngoặc đóng chốt mọi thứ trong ngoặc ⇒ nội dung ngoặc thành một nút duy nhất, hoạt động như một toán hạng. Mỗi lần chốt lấy 2 nút liền kề trên stack (theo đúng thứ tự trái–phải) ⇒ inorder giữ thứ tự, mọi nút trong có đúng 2 con.

## 11. Phân tích Time / Space

Gọi `n = len(s)` (≤ 100), `k` = số toán tử.

| Cách | Time | Space |
|---|---|---|
| 1. Thử mọi cây | O(C_k · n), C_k ~ 4^k | O(C_k · n) để giữ các cây |
| 2. Chia để trị | O(n²) worst | O(n) |
| 3. Recursive descent | O(n) | O(n) (đệ quy theo độ lồng ngoặc) |
| 4. Hai ngăn xếp | O(n) | O(n) |

Đo nhanh trên biểu thức dài tối đa:

In [17]:
def gen_expr(rnd, max_len):
    """Sinh ngẫu nhiên biểu thức hợp lệ (toán hạng 1 chữ số) có độ dài <= max_len."""
    def rec(budget):
        if budget < 5 or rnd.random() < 0.25:
            if budget >= 3 and rnd.random() < 0.15:
                return "(" + str(rnd.randint(0, 9)) + ")"
            return str(rnd.randint(0, 9))
        if rnd.random() < 0.2 and budget >= 7:
            return "(" + rec(budget - 2) + ")"
        left_b = rnd.randint(1, budget - 2)
        return rec(left_b) + rnd.choice("+-*/") + rec(budget - 1 - left_b)
    s = rec(max_len)
    return s if len(s) <= max_len else str(rnd.randint(0, 9))

rnd = random.Random(1)
long_s = max((gen_expr(rnd, 100) for _ in range(200)), key=len)
print(len(long_s), long_s)
for f in [exp_tree_split, exp_tree_rd, exp_tree_stacks]:
    t0 = time.perf_counter()
    for _ in range(200):
        tr = f(long_s)
    print(f"{f.__name__:16s} 200 lần: {time.perf_counter() - t0:.3f}s, hợp lệ: {valid_strong(long_s, tr)}")

59 3*3*(4)-0*(4)-4+(3*1)/(7*0+0)/2-3*9+4*7+0+0/2+3-((0))/8*2+0
exp_tree_split   200 lần: 0.014s, hợp lệ: True
exp_tree_rd      200 lần: 0.004s, hợp lệ: True
exp_tree_stacks  200 lần: 0.004s, hợp lệ: True


## 12. Bộ test

Vì nhiều đáp án, mọi test dùng `valid_strong` (bao gồm `valid_basic`). Gồm: ví dụ đề, edge case (1 chữ số, ngoặc lồng nhiều tầng, toàn trừ, toàn chia, chia cho 0 trong biểu thức), và biểu thức ngẫu nhiên.

In [18]:
ALL = [exp_tree_split, exp_tree_rd, exp_tree_stacks, lambda x: Solution().expTree(x)]
fixed = [s for s, _ in examples] + [
    "7", "(7)", "((((7))))", "1-2-3-4", "8/2/2", "2*(3+4)*5", "(1+2)*(3-4)/(5+6)",
    "9-(8-(7-(6-5)))", "1/(2-2)", "0*0-0", "1+2*3-4/5*6+7", "((1))+((2))*((3))",
]
for s in fixed:
    for f in ALL:
        tr = f(s)
        assert valid_strong(s, tr), (f, s, tree_to_list(tr))
print("Ca cố định:", len(fixed), "x", len(ALL), "cách: OK")

# Ví dụ 1 có duy nhất 1 cây hợp lệ -> mọi cách phải cho đúng chuỗi mẫu
for f in ALL:
    assert tree_to_list(f("3*4-2*5")) == "[-,*,*,3,4,2,5]"
print("Ví dụ 1 trùng đáp án mẫu ở mọi cách")

Ca cố định: 15 x 4 cách: OK
Ví dụ 1 trùng đáp án mẫu ở mọi cách


In [19]:
rnd = random.Random(2024)
cnt = 0
for _ in range(800):
    s = gen_expr(rnd, rnd.randint(1, 100))
    for f in ALL:
        assert valid_strong(s, f(s), trials=3), (f, s)
    cnt += 1
# Đối chiếu với brute force trên biểu thức ngắn (ít toán tử)
for _ in range(150):
    s = gen_expr(rnd, rnd.randint(1, 13))
    b = exp_tree_brute(s)
    assert b is not None and valid_strong(s, b)
    assert eval_tree(b) == eval_tree(Solution().expTree(s))
    cnt += 1
print("Ngẫu nhiên:", cnt, "biểu thức OK")

Ngẫu nhiên: 950 biểu thức OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Bỏ qua thứ tự ưu tiên: dựng cây từ trái sang phải

Cây vẫn có **inorder đúng** nhưng **tính sai** — đây chính là loại cây "không hợp lệ" mà ví dụ 2 của đề cảnh báo.

In [20]:
def wrong_left_to_right(s):
    plain = [c for c in s if c not in "()"]
    node = Node(plain[0])
    for k in range(1, len(plain), 2):
        node = Node(plain[k], node, Node(plain[k + 1]))   # bỏ qua ưu tiên và ngoặc
    return node

for s in ["3*4-2*5", "2-3/(5*2)+1"]:
    tr = wrong_left_to_right(s)
    print(f"{s}: inorder khớp = {inorder(tr) == s.replace('(', '').replace(')', '')}, "
          f"giá trị cây = {eval_tree(tr)}, giá trị đúng = {ref_value(s)}, hợp lệ = {valid_basic(s, tr)}")

3*4-2*5: inorder khớp = True, giá trị cây = 50, giá trị đúng = 2, hợp lệ = False
2-3/(5*2)+1: inorder khớp = True, giá trị cây = 3/5, giá trị đúng = 27/10, hợp lệ = False


**Vì sao sai:** `((3*4)-2)*5 = 50 ≠ 2`. Inorder chỉ ghi lại **thứ tự ký tự**, không ghi lại **cấu trúc**. **Sửa:** dùng một trong các Cách 2–4.

### Lỗi 2 — Chia để trị chọn toán tử bên TRÁI nhất (sai kết hợp)

In [21]:
def wrong_leftmost(s):
    match = match_parens(s)
    def build(l, r):
        while s[l] == "(" and match[l] == r - 1:
            l += 1; r -= 1
        if r - l == 1:
            return Node(s[l])
        depth, pos_add, pos_mul = 0, -1, -1
        for k in range(l, r):
            c = s[k]
            if c == "(": depth += 1
            elif c == ")": depth -= 1
            elif depth == 0:
                if c in "+-" and pos_add == -1: pos_add = k     # SAI: giữ cái đầu tiên
                elif c in "*/" and pos_mul == -1: pos_mul = k
        k = pos_add if pos_add != -1 else pos_mul
        return Node(s[k], build(l, k), build(k + 1, r))
    return build(0, len(s))

for s in ["1-2-3", "8/2/2", "1+2+3"]:
    tr = wrong_leftmost(s)
    print(f"{s}: cây {to_paren(tr)} = {eval_tree(tr)}, đúng = {ref_value(s)}, hợp lệ = {valid_basic(s, tr)}")

1-2-3: cây (1-(2-3)) = 2, đúng = -4, hợp lệ = False
8/2/2: cây (8/(2/2)) = 8, đúng = 2, hợp lệ = False
1+2+3: cây (1+(2+3)) = 6, đúng = 6, hợp lệ = True


**Vì sao sai:** `1-(2-3) = 2` nhưng `1-2-3 = -4`; `8/(2/2) = 8` nhưng `8/2/2 = 2`. Với `+` thì vô tình đúng giá trị (phép cộng có tính kết hợp) — vì thế lỗi này dễ lọt nếu chỉ test với `+`. **Sửa:** giữ toán tử **bên phải nhất** (ghi đè `pos_add = k`).

### Lỗi 3 — Hai stack dùng `>` thay vì `>=` khi chốt

In [22]:
def wrong_strict(s):
    prec = {"+": 1, "-": 1, "*": 2, "/": 2}
    nodes, ops = [], []
    def red():
        op = ops.pop(); r = nodes.pop(); l = nodes.pop(); nodes.append(Node(op, l, r))
    for c in s:
        if c.isdigit(): nodes.append(Node(c))
        elif c == "(": ops.append(c)
        elif c == ")":
            while ops[-1] != "(": red()
            ops.pop()
        else:
            while ops and ops[-1] != "(" and prec[ops[-1]] > prec[c]:   # SAI: '>'
                red()
            ops.append(c)
    while ops: red()
    return nodes[-1]

for s in ["1-2-3", "8/2/2", "3*4-2*5"]:
    tr = wrong_strict(s)
    print(f"{s}: cây {to_paren(tr)} = {eval_tree(tr)}, đúng = {ref_value(s)}, hợp lệ = {valid_basic(s, tr)}")

1-2-3: cây (1-(2-3)) = 2, đúng = -4, hợp lệ = False
8/2/2: cây (8/(2/2)) = 8, đúng = 2, hợp lệ = False
3*4-2*5: cây ((3*4)-(2*5)) = 2, đúng = 2, hợp lệ = True


**Vì sao sai:** với `>`, toán tử cùng mức không được chốt ⇒ chúng tích lũy rồi được chốt từ **phải sang trái** ⇒ kết hợp phải. Ưu tiên khác mức (`3*4-2*5`) vẫn đúng nên lỗi dễ bị che. **Sửa:** `>=`.

### Lỗi 4 — Đảo thứ tự khi pop hai nút

In [23]:
def wrong_pop_order(s):
    prec = {"+": 1, "-": 1, "*": 2, "/": 2}
    nodes, ops = [], []
    def red():
        op = ops.pop(); l = nodes.pop(); r = nodes.pop()      # SAI: nút pop trước là con PHẢI
        nodes.append(Node(op, l, r))
    for c in s:
        if c.isdigit(): nodes.append(Node(c))
        elif c == "(": ops.append(c)
        elif c == ")":
            while ops[-1] != "(": red()
            ops.pop()
        else:
            while ops and ops[-1] != "(" and prec[ops[-1]] >= prec[c]: red()
            ops.append(c)
    while ops: red()
    return nodes[-1]

s = "7-2"
tr = wrong_pop_order(s)
print(f"{s}: inorder {inorder(tr)!r} (cần {s!r}), giá trị {eval_tree(tr)} (cần {ref_value(s)}), hợp lệ = {valid_basic(s, tr)}")

7-2: inorder '2-7' (cần '7-2'), giá trị -5 (cần 5), hợp lệ = False


**Vì sao sai:** stack là LIFO — phần tử lấy ra **đầu tiên** là toán hạng **bên phải**. Đảo lại thì inorder thành `2-7`, toán hạng đổi thứ tự. **Sửa:** `right = nodes.pop(); left = nodes.pop()`.

### Lỗi 5 — Chia để trị quên bỏ qua ngoặc (không theo dõi `depth`)

In [24]:
def wrong_no_depth(s):
    match = match_parens(s)
    def build(l, r):
        while s[l] == "(" and match[l] == r - 1:
            l += 1; r -= 1
        if r - l == 1:
            return Node(s[l])
        pos_add = pos_mul = -1
        for k in range(l, r):                    # SAI: không theo dõi depth
            if s[k] in "+-": pos_add = k
            elif s[k] in "*/": pos_mul = k
        k = pos_add if pos_add != -1 else pos_mul
        return Node(s[k], build(l, k), build(k + 1, r))
    return build(0, len(s))

def wrong_strip(s):
    def build(l, r):
        while s[l] == "(" and s[r - 1] == ")":   # SAI: không kiểm tra hai ngoặc có cùng cặp
            l += 1; r -= 1
        if r - l == 1:
            return Node(s[l])
        depth, pos_add, pos_mul = 0, -1, -1
        for k in range(l, r):
            c = s[k]
            if c == "(": depth += 1
            elif c == ")": depth -= 1
            elif depth == 0:
                if c in "+-": pos_add = k
                elif c in "*/": pos_mul = k
        k = pos_add if pos_add != -1 else pos_mul
        return Node(s[k], build(l, k), build(k + 1, r))
    return build(0, len(s))

def show(f, s):
    try:
        tr = f(s)
    except Exception as e:                        # bắt lỗi để notebook chạy tiếp
        print(f"{f.__name__}({s!r}) -> văng {type(e).__name__}")
        return
    print(f"{f.__name__}({s!r}) -> cây {to_paren(tr)} = {eval_tree(tr)}, đúng = {ref_value(s)}, hợp lệ = {valid_basic(s, tr)}")

for s in ["1+2*(3-4)", "(1+2)*3"]:
    show(wrong_no_depth, s)
show(wrong_strip, "(1+2)*(3+4)")
show(exp_tree_split, "(1+2)*(3+4)")

wrong_no_depth('1+2*(3-4)') -> văng RecursionError
wrong_no_depth('(1+2)*3') -> văng RecursionError
wrong_strip('(1+2)*(3+4)') -> văng RecursionError
exp_tree_split('(1+2)*(3+4)') -> cây ((1+2)*(3+4)) = 21, đúng = 21, hợp lệ = True


**Vì sao sai:** `wrong_no_depth` chọn toán tử **bên trong ngoặc** làm gốc: với `1+2*(3-4)` nó chọn `-` (cuối cùng, nhưng nằm trong ngoặc) ⇒ hai nửa `1+2*(3` và `4)` không còn là biểu thức hợp lệ ⇒ chương trình hỏng hoặc cho cây vô nghĩa. `wrong_strip` thấy `(` ở đầu và `)` ở cuối `(1+2)*(3+4)` nhưng hai ngoặc này **không cùng cặp** ⇒ bóc nhầm thành `1+2)*(3+4` và hỏng cấu trúc. **Sửa:** đếm `depth` và dùng bảng `match` như Cách 2.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- **Biểu thức trung tố + ưu tiên + ngoặc** ⇒ hai stack (shunting-yard) hoặc recursive descent. Đây là cùng khung với LC 224/227/772 (Basic Calculator I/II/III) — chỉ khác là "chốt" tạo **nút** thay vì tính **số**.
- Nhớ câu thần chú: **"gặp toán tử: chốt hết những gì ưu tiên ≥ nó"** (cho toán tử kết hợp trái).
- Pop hai toán hạng: **phải trước, trái sau**.
- Bài "trả về bất kỳ đáp án hợp lệ" ⇒ khi tự test, viết **validator**; nếu kiểm theo giá trị, thử thêm **thay toán hạng ngẫu nhiên** để tránh nhầm may mắn.
- Muốn chắc kết hợp trái: luôn test với `-` và `/` liên tiếp (`1-2-3`, `8/2/2`), không chỉ `+`.

## 15. Biến thể của bài

1. **LC 224 / 227 / 772 — Basic Calculator:** tính giá trị thay vì dựng cây (cùng khung hai stack).
2. **LC 1628 — Design an Expression Tree With Evaluate Function:** dựng cây từ **hậu tố** (postfix) — chỉ cần một stack.
3. **LC 150 — Evaluate Reverse Polish Notation.**
4. **LC 1612 — Check If Two Expression Trees are Equivalent** (chỉ có `+`).
5. **Toán hạng nhiều chữ số / biến** — đọc liên tiếp các chữ số thành một token.
6. **In biểu thức với số ngoặc tối thiểu** từ một cây — ngược của bài này.
7. **Thêm toán tử lũy thừa `^` (kết hợp phải)** — điều kiện chốt đổi thành `>` riêng cho `^`.

In [25]:
# Minh họa biến thể 7: '^' kết hợp phải, ưu tiên cao nhất
def exp_tree_with_pow(s):
    prec = {"+": 1, "-": 1, "*": 2, "/": 2, "^": 3}
    right_assoc = {"^"}
    nodes, ops = [], []
    def red():
        op = ops.pop(); r = nodes.pop(); l = nodes.pop(); nodes.append(Node(op, l, r))
    for c in s:
        if c.isdigit(): nodes.append(Node(c))
        elif c == "(": ops.append(c)
        elif c == ")":
            while ops[-1] != "(": red()
            ops.pop()
        else:
            while ops and ops[-1] != "(" and (prec[ops[-1]] > prec[c] or
                                             (prec[ops[-1]] == prec[c] and c not in right_assoc)):
                red()
            ops.append(c)
    while ops: red()
    return nodes[-1]

print(to_paren(exp_tree_with_pow("2^3^2")), to_paren(exp_tree_with_pow("1+2*3^2-4")))

(2^(3^2)) ((1+(2*(3^2)))-4)


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Thử mọi cây | sinh cây Catalan, lọc bằng validator | hàm mũ | hàm mũ | hiểu đề, đối chiếu biểu thức ngắn |
| 2. Chia để trị | gốc = toán tử ngoài ngoặc, ưu tiên thấp nhất, phải nhất | O(n²) | O(n) | trực quan nhất, `n` nhỏ |
| 3. Recursive descent | văn phạm expr/term/factor | O(n) | O(n) | dễ mở rộng (thêm toán tử, hàm) |
| 4. Hai stack | shunting-yard dựng nút | O(n) | O(n) | **nộp bài**, không đệ quy |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** Toán tử ở **gốc** của cây hợp lệ cho mỗi biểu thức là gì? (a) `1+2*3`; (b) `(1+2)*3`; (c) `1*2-3*4+5`; (d) `9/(3/(1+2))`.

**Bài 2 (tính tay):** Viết level-order (kiểu LeetCode) của cây do Cách 4 dựng cho `"1-2-3"` và cho `"(1-2)*3+4"`.

**Bài 3 (điền TODO):** Hoàn thiện `postfix(root)` trả về chuỗi hậu tố (duyệt hậu thứ tự) của cây biểu thức.

**Bài 4 (tự viết lại):** Viết `min_paren(root)` in cây thành biểu thức trung tố với **ít ngoặc nhất** sao cho đọc lại (theo quy tắc thường) vẫn ra đúng cây/giá trị.

**Bài 5 (mini challenge):** Viết `exp_tree_multi(s)` cho phép **toán hạng nhiều chữ số** (ví dụ `"12*(30-4)/2"`), lá chứa cả số.

In [26]:
# Bài 3 — TODO (chỉ định nghĩa, không gọi)
def postfix(root):
    # TODO: nếu là lá trả root.val
    # TODO: ngược lại trả postfix(trái) + postfix(phải) + root.val
    pass

In [27]:
# Bài 4, 5 — khung tự viết (chỉ định nghĩa, không gọi)
def min_paren(root):
    # TODO
    pass

def exp_tree_multi(s):
    # TODO
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** Mức 1: tìm toán tử tính cuối cùng. Mức 2: ngoài ngoặc, ưu tiên thấp, bên phải nhất. Mức 3: (a) `+`; (b) `*`; (c) `+` (cái cuối); (d) `/` (cái đầu tiên — cái thứ hai nằm trong ngoặc).

**Bài 2.** Mức 1: `1-2-3 = (1-2)-3`. Mức 2: gốc `-`, con trái là `(1-2)`, con phải `3`. Mức 3: `[-,-,3,1,2]` và `[+,*,4,-,3,null,null,1,2]`.

**Bài 3.** Mức 1: hậu thứ tự = trái, phải, gốc. Mức 2: lá không có con. Mức 3: `"34*25*-"` cho ví dụ 1.

**Bài 4.** Mức 1: con **trái** cần ngoặc khi ưu tiên của nó **thấp hơn** cha (`(a+b)*c`). Mức 2: con **phải** cần ngoặc khi ưu tiên thấp hơn cha, hoặc **cùng mức mà cha là `-` hoặc `/`** (vì `a-(b-c) ≠ a-b-c`, `a/(b/c) ≠ a/b/c`). Mức 3: con phải cùng mức dưới cha `+` hoặc `*` **không** cần ngoặc vì tính kết hợp: `a+(b-c) = a+b-c`, `a*(b/c) = a*b/c` (đọc lại ra cây khác nhưng **cùng giá trị** — được chấp nhận như ví dụ 3); kiểm tra bằng `valid_strong` (thay số ngẫu nhiên).

**Bài 5.** Mức 1: khi gặp chữ số, đọc tiếp các chữ số liền sau. Mức 2: duyệt bằng chỉ số `i` thay vì `for c in s`. Mức 3: phần còn lại giống hệt Cách 4.

## 19. Đáp án tham khảo cho bài tập

In [28]:
# Bài 1, 2
roots = [Solution().expTree(x).val for x in ["1+2*3", "(1+2)*3", "1*2-3*4+5", "9/(3/(1+2))"]]
print("Bài 1:", roots)
assert roots == ["+", "*", "+", "/"]
b2 = [tree_to_list(Solution().expTree(x)) for x in ["1-2-3", "(1-2)*3+4"]]
print("Bài 2:", b2)
assert b2 == ["[-,-,3,1,2]", "[+,*,4,-,3,null,null,1,2]"]

Bài 1: ['+', '*', '+', '/']
Bài 2: ['[-,-,3,1,2]', '[+,*,4,-,3,null,null,1,2]']


In [29]:
# Bài 3
def postfix_sol(root):
    if root.left is None and root.right is None:
        return root.val
    return postfix_sol(root.left) + postfix_sol(root.right) + root.val

def eval_postfix(p):
    st = []
    for c in p:
        if c.isdigit():
            st.append(Fraction(int(c)))
        else:
            b = st.pop(); a = st.pop()
            if a is None or b is None or (c == "/" and b == 0):
                st.append(None); continue
            st.append(a + b if c == "+" else a - b if c == "-" else a * b if c == "*" else a / b)
    return st[0]

assert postfix_sol(Solution().expTree("3*4-2*5")) == "34*25*-"
rnd = random.Random(3)
for _ in range(300):
    s = gen_expr(rnd, rnd.randint(1, 60))
    assert eval_postfix(postfix_sol(Solution().expTree(s))) == ref_value(s)
print("Bài 3: OK")

Bài 3: OK


In [30]:
# Bài 4
def min_paren_sol(root):
    prec = {"+": 1, "-": 1, "*": 2, "/": 2}
    def go(node):
        if node.left is None:
            return node.val
        L, R = go(node.left), go(node.right)
        p = prec[node.val]
        if node.left.left is not None and prec[node.left.val] < p:
            L = "(" + L + ")"
        if node.right.left is not None:
            q = prec[node.right.val]
            if q < p or (q == p and node.val in "-/"):
                R = "(" + R + ")"
        return L + node.val + R
    return go(root)

rnd = random.Random(4)
for _ in range(400):
    s = gen_expr(rnd, rnd.randint(1, 60))
    tr = Solution().expTree(s)
    out = min_paren_sol(tr)
    assert out.replace("(", "").replace(")", "") == s.replace("(", "").replace(")", "")
    assert valid_strong(out, tr, trials=3)          # đọc lại ra cùng cấu trúc (theo giá trị ngẫu nhiên)
print("Bài 4: OK;", min_paren_sol(Solution().expTree("((1+2))*(3-(4-5))/(6*7)")))

Bài 4: OK; (1+2)*(3-(4-5))/(6*7)


In [31]:
# Bài 5
def exp_tree_multi_sol(s):
    prec = {"+": 1, "-": 1, "*": 2, "/": 2}
    nodes, ops = [], []
    def red():
        op = ops.pop(); r = nodes.pop(); l = nodes.pop(); nodes.append(Node(op, l, r))
    i, n = 0, len(s)
    while i < n:
        c = s[i]
        if c.isdigit():
            j = i
            while j < n and s[j].isdigit():
                j += 1
            nodes.append(Node(s[i:j])); i = j; continue
        if c == "(":
            ops.append(c)
        elif c == ")":
            while ops[-1] != "(": red()
            ops.pop()
        else:
            while ops and ops[-1] != "(" and prec[ops[-1]] >= prec[c]: red()
            ops.append(c)
        i += 1
    while ops: red()
    return nodes[-1]

def eval_multi(node):
    if node.left is None:
        return Fraction(int(node.val))
    a, b = eval_multi(node.left), eval_multi(node.right)
    return a + b if node.val == "+" else a - b if node.val == "-" else a * b if node.val == "*" else a / b

tr = exp_tree_multi_sol("12*(30-4)/2")
print("Bài 5:", tree_to_list(tr), "=", eval_multi(tr))
assert eval_multi(tr) == Fraction(156)
rnd = random.Random(5)
for _ in range(300):
    parts = [str(rnd.randint(1, 999))]
    for _ in range(rnd.randint(0, 6)):
        parts += [rnd.choice("+-*/"), str(rnd.randint(1, 999))]
    s = "".join(parts)
    assert eval_multi(exp_tree_multi_sol(s)) == eval(re.sub(r"\d+", lambda m: f"F({m.group()})", s), {"F": Fraction})
print("Bài 5: OK")

Bài 5: [/,*,2,12,-,null,null,null,null,30,4] = 156
Bài 5: OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template hai stack dựng cây:**

```python
prec = {'+': 1, '-': 1, '*': 2, '/': 2}
nodes, ops = [], []
def reduce_once():
    op = ops.pop(); right = nodes.pop(); left = nodes.pop()
    nodes.append(Node(op, left, right))
for c in s:
    if c.isdigit(): nodes.append(Node(c))
    elif c == '(': ops.append(c)
    elif c == ')':
        while ops[-1] != '(': reduce_once()
        ops.pop()
    else:
        while ops and ops[-1] != '(' and prec[ops[-1]] >= prec[c]: reduce_once()
        ops.append(c)
while ops: reduce_once()
return nodes[-1]
```

**Checklist:**
- [ ] Chốt khi ưu tiên đỉnh **≥** (không phải `>`) — kết hợp trái?
- [ ] Pop **phải trước, trái sau**?
- [ ] Gặp `)` chốt tới `(` rồi **bỏ** `(`?
- [ ] Không chốt xuyên qua `(` khi gặp toán tử?
- [ ] Đã test `1-2-3`, `8/2/2`, `3*4-2*5`, ngoặc lồng nhiều tầng, biểu thức một chữ số `"7"`, `"(7)"`?
- [ ] Đối chiếu tên lớp `Node` / method `expTree` với đề Premium nếu có tài khoản.